# B4: Cohort Temporal Analysis — Group Dynamics in Depression

This notebook validates **RFC-007** features on the eRisk dataset:

| Feature | Question |
|---------|----------|
| `cohort_drift` | Does the depressed cohort drift differently from controls? |
| `temporal_join` | When do depressed users converge with distress anchors? |
| `counterfactual_trajectory` | What would have happened without the detected change? |

**Dataset**: eRisk 2022 — 225K posts, 466 users, MentalRoBERTa D=768.

In [1]:
import chronos_vector as cvx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
import json, time, os, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
EMB_DIR = f'{DATA_DIR}/embeddings'

C_DEP = '#e74c3c'
C_CTL = '#3498db'
C_ACCENT = '#2ecc71'

## 1. Data Loading

Reuse cached CVX index + parquet embeddings from B1/B2.

In [2]:
df_full = pd.read_parquet(f'{EMB_DIR}/erisk_mental_embeddings.parquet')
emb_cols = [c for c in df_full.columns if c.startswith('emb_')]
D = len(emb_cols)

dep_users = df_full[df_full['label'] == 'depression']['user_id'].unique()
ctl_users = df_full[df_full['label'] == 'control']['user_id'].unique()
np.random.seed(42)
n_ctl = min(len(dep_users), len(ctl_users))
ctl_sample = np.random.choice(ctl_users, size=n_ctl, replace=False)
selected_users = np.concatenate([dep_users, ctl_sample])
df = df_full[df_full['user_id'].isin(selected_users)].copy()

user_to_id = {u: i for i, u in enumerate(sorted(df['user_id'].unique()))}
df['entity_id'] = df['user_id'].map(user_to_id).astype(np.uint64)
label_map = dict(zip(df['user_id'], df['label']))

print(f'{len(df):,} posts, {df["user_id"].nunique()} users (dep={len(dep_users)}, ctl={n_ctl}), D={D}')

225,962 posts, 466 users (dep=233, ctl=233), D=768


In [3]:
# Load cached index
INDEX_PATH = f'{DATA_DIR}/cache/erisk_index.cvx'
t0 = time.perf_counter()
index = cvx.TemporalIndex.load(INDEX_PATH)
print(f'Index loaded in {time.perf_counter() - t0:.1f}s ({len(index):,} points)')

Index loaded in 0.6s (225,962 points)


## 2. Build Per-User Trajectories

Extract anchor-projected trajectories for all users. We reuse the DSM-5 anchors from B2.

In [4]:
# Load DSM-5 anchor vectors from B2 cache
ANCHOR_CACHE = f'{DATA_DIR}/cache/dsm5_anchors.npz'
data = np.load(ANCHOR_CACHE, allow_pickle=True)
anchor_vectors = data['anchor_vectors'].item()
healthy_vector = data['healthy_vector']

anchor_names = list(anchor_vectors.keys()) + ['healthy']
anchors_np = np.array([anchor_vectors[k] for k in anchor_vectors.keys()] + [healthy_vector])
K = len(anchor_names)
print(f'{K} anchors: {anchor_names}')

10 anchors: ['depressed_mood', 'anhedonia', 'sleep_disturbance', 'fatigue', 'worthlessness', 'concentration', 'suicidal_ideation', 'appetite', 'psychomotor', 'healthy']


In [5]:
# Build per-user trajectories as (entity_id, [(timestamp, vector), ...])
user_trajectories = {}
for uid, grp in df.sort_values('timestamp').groupby('user_id'):
    eid = user_to_id[uid]
    traj = []
    for _, row in grp.iterrows():
        vec = row[emb_cols].values.astype(np.float32).tolist()
        traj.append((int(row['timestamp'].value), vec))
    user_trajectories[uid] = (eid, traj)

print(f'{len(user_trajectories)} user trajectories built')
print(f'Mean trajectory length: {np.mean([len(t[1]) for t in user_trajectories.values()]):.0f}')

466 user trajectories built
Mean trajectory length: 485


In [6]:
# Project all trajectories to anchor space
user_anchor_trajs = {}
anchors_list = anchors_np.tolist()

for uid, (eid, traj) in user_trajectories.items():
    projected = cvx.project_to_anchors(traj, anchors_list, 'cosine')
    user_anchor_trajs[uid] = projected

print(f'Projected {len(user_anchor_trajs)} trajectories to {K}-dim anchor space')

Projected 466 trajectories to 10-dim anchor space


---
## 3. Experiment 1: Cohort Drift Analysis

**Hypothesis**: The depressed cohort drifts differently from controls.
We measure cohort-level drift between the first and second half of each user's timeline.

In [7]:
# Define time boundaries: first half vs second half of dataset
all_timestamps = df["timestamp"].values.astype("int64")  # nanoseconds
t_mid = int(np.median(all_timestamps))
t_start = int(all_timestamps.min())
t_end = int(all_timestamps.max())
print(f"Time range: {t_start} to {t_end}, midpoint: {t_mid}")


Time range: -9223372036854775808 to 1640035530000000, midpoint: 1564299992000000


In [8]:
# Build cohort trajectories in anchor space
dep_trajs = [(user_to_id[uid], user_anchor_trajs[uid]) 
             for uid in dep_users if uid in user_anchor_trajs]
ctl_trajs = [(user_to_id[uid], user_anchor_trajs[uid]) 
             for uid in ctl_sample if uid in user_anchor_trajs]

print(f'Depression cohort: {len(dep_trajs)} users')
print(f'Control cohort: {len(ctl_trajs)} users')

Depression cohort: 233 users
Control cohort: 233 users


In [9]:
# Compute cohort drift for each group
dep_report = cvx.cohort_drift(dep_trajs, t_start, t_end, top_n=K)
ctl_report = cvx.cohort_drift(ctl_trajs, t_start, t_end, top_n=K)

print('=== DEPRESSION COHORT ===')
print(f'  N entities:        {dep_report["n_entities"]}')
print(f'  Mean drift (L2):   {dep_report["mean_drift_l2"]:.4f}')
print(f'  Median drift:      {dep_report["median_drift_l2"]:.4f}')
print(f'  Std drift:         {dep_report["std_drift_l2"]:.4f}')
print(f'  Centroid drift:    {dep_report["centroid_l2"]:.4f}')
print(f'  Dispersion change: {dep_report["dispersion_change"]:+.4f}')
print(f'  Convergence score: {dep_report["convergence_score"]:.4f}')
print(f'  Outliers:          {len(dep_report["outliers"])}')
print()
print('=== CONTROL COHORT ===')
print(f'  N entities:        {ctl_report["n_entities"]}')
print(f'  Mean drift (L2):   {ctl_report["mean_drift_l2"]:.4f}')
print(f'  Median drift:      {ctl_report["median_drift_l2"]:.4f}')
print(f'  Std drift:         {ctl_report["std_drift_l2"]:.4f}')
print(f'  Centroid drift:    {ctl_report["centroid_l2"]:.4f}')
print(f'  Dispersion change: {ctl_report["dispersion_change"]:+.4f}')
print(f'  Convergence score: {ctl_report["convergence_score"]:.4f}')
print(f'  Outliers:          {len(ctl_report["outliers"])}')

=== DEPRESSION COHORT ===
  N entities:        233
  Mean drift (L2):   0.0478
  Median drift:      0.0366
  Std drift:         0.0425
  Centroid drift:    0.0104
  Dispersion change: +0.0018
  Convergence score: 0.2073
  Outliers:          10

=== CONTROL COHORT ===
  N entities:        233
  Mean drift (L2):   0.0599
  Median drift:      0.0479
  Std drift:         0.0457
  Centroid drift:    0.0051
  Dispersion change: +0.0006
  Convergence score: 0.0508
  Outliers:          9


In [10]:
# Visualize cohort comparison
metrics = ['mean_drift_l2', 'centroid_l2', 'convergence_score']
dep_vals = [dep_report[m] for m in metrics]
ctl_vals = [ctl_report[m] for m in metrics]

fig = go.Figure(data=[
    go.Bar(name='Depression', x=metrics, y=dep_vals, marker_color=C_DEP),
    go.Bar(name='Control', x=metrics, y=ctl_vals, marker_color=C_CTL),
])
fig.update_layout(
    title='Cohort Drift Comparison: Depression vs Control',
    barmode='group', height=400,
    yaxis_title='Value',
)
fig.show()

In [11]:
# Dispersion change visualization
fig = go.Figure(data=[
    go.Bar(
        x=['Depression', 'Control'],
        y=[dep_report['dispersion_change'], ctl_report['dispersion_change']],
        marker_color=[C_DEP, C_CTL],
        text=[f'{dep_report["dispersion_change"]:+.4f}', f'{ctl_report["dispersion_change"]:+.4f}'],
        textposition='outside',
    )
])
fig.update_layout(
    title='Dispersion Change: Positive = Diverging, Negative = Converging',
    yaxis_title='Dispersion Δ', height=350,
)
fig.show()

---
## 4. Experiment 2: Temporal Join — Convergence with Distress Anchors

**Hypothesis**: Depressed users spend more time near distress anchors (depressed_mood, worthlessness, suicidal_ideation) than controls.

We compute temporal joins between each user's anchor-projected trajectory and the distress pole.

In [12]:
# Create a "distress trajectory" — constant at the distress pole in anchor space
# In anchor space, the distress pole has distance ~0 to distress anchors
# and distance ~1 to healthy anchor
distress_pole = [0.0] * len([k for k in anchor_vectors.keys()]) + [1.0]  # close to all symptoms, far from healthy

# Use the time range of each user as the distress trajectory timestamps
window_us = 30 * 86_400 * 1_000_000  # 30-day windows
epsilon = 0.8  # anchor cosine distances are small; tuned from 0.3  # distance threshold in anchor space

dep_join_stats = []
ctl_join_stats = []

for uid in dep_users:
    if uid not in user_anchor_trajs or len(user_anchor_trajs[uid]) < 2:
        continue
    traj = user_anchor_trajs[uid]
    # Create matching distress trajectory
    distress_traj = [(ts, distress_pole) for ts, _ in traj]
    try:
        windows = cvx.temporal_join(traj, distress_traj, epsilon, window_us)
        total_convergence_time = sum(e - s for s, e, _, _, _, _ in windows)
        dep_join_stats.append({
            'user_id': uid, 'n_windows': len(windows),
            'total_convergence_us': total_convergence_time,
        })
    except Exception:
        dep_join_stats.append({'user_id': uid, 'n_windows': 0, 'total_convergence_us': 0})

for uid in ctl_sample:
    if uid not in user_anchor_trajs or len(user_anchor_trajs[uid]) < 2:
        continue
    traj = user_anchor_trajs[uid]
    distress_traj = [(ts, distress_pole) for ts, _ in traj]
    try:
        windows = cvx.temporal_join(traj, distress_traj, epsilon, window_us)
        total_convergence_time = sum(e - s for s, e, _, _, _, _ in windows)
        ctl_join_stats.append({
            'user_id': uid, 'n_windows': len(windows),
            'total_convergence_us': total_convergence_time,
        })
    except Exception:
        ctl_join_stats.append({'user_id': uid, 'n_windows': 0, 'total_convergence_us': 0})

dep_df = pd.DataFrame(dep_join_stats)
ctl_df = pd.DataFrame(ctl_join_stats)

print(f'Depression: mean {dep_df["n_windows"].mean():.1f} convergence windows, '
      f'mean time {dep_df["total_convergence_us"].mean()/86_400e6:.1f} days')
print(f'Control:    mean {ctl_df["n_windows"].mean():.1f} convergence windows, '
      f'mean time {ctl_df["total_convergence_us"].mean()/86_400e6:.1f} days')

Depression: mean 0.0 convergence windows, mean time 0.0 days
Control:    mean 0.0 convergence windows, mean time 0.0 days


In [13]:
# Visualize convergence window distribution
fig = go.Figure()
fig.add_trace(go.Histogram(x=dep_df['n_windows'], name='Depression',
                           marker_color=C_DEP, opacity=0.7))
fig.add_trace(go.Histogram(x=ctl_df['n_windows'], name='Control',
                           marker_color=C_CTL, opacity=0.7))
fig.update_layout(
    title='Distribution of Distress Convergence Windows per User',
    xaxis_title='Number of convergence windows',
    yaxis_title='Count', barmode='overlay', height=400,
)
fig.show()

In [14]:
# Use temporal join features for classification
all_join_stats = pd.concat([
    dep_df.assign(label=1),
    ctl_df.assign(label=0),
])

X_join = all_join_stats[['n_windows', 'total_convergence_us']].values
y_join = all_join_stats['label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_join)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1s, aucs = [], []
for train_idx, test_idx in skf.split(X_scaled, y_join):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_scaled[train_idx], y_join[train_idx])
    y_pred = clf.predict(X_scaled[test_idx])
    y_prob = clf.predict_proba(X_scaled[test_idx])[:, 1]
    f1s.append(f1_score(y_join[test_idx], y_pred))
    aucs.append(roc_auc_score(y_join[test_idx], y_prob))

print(f'Temporal Join features → Depression classification:')
print(f'  F1:  {np.mean(f1s):.3f} ± {np.std(f1s):.3f}')
print(f'  AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}')

Temporal Join features → Depression classification:
  F1:  0.265 ± 0.324
  AUC: 0.500 ± 0.000


---
## 5. Experiment 3: Counterfactual Trajectories

For users with detected change points, we ask: **what would have happened if the change hadn't occurred?**

We extrapolate the pre-change anchor-projected trajectory and measure how much the actual trajectory diverged.

In [15]:
# Detect change points for depressed users, then compute counterfactuals
counterfactual_results = []

for uid in dep_users[:50]:  # first 50 depressed users
    if uid not in user_anchor_trajs:
        continue
    traj = user_anchor_trajs[uid]
    if len(traj) < 10:
        continue
    
    # Detect change points
    eid = user_to_id[uid]
    cps = cvx.detect_changepoints(eid, traj, penalty=2.0, min_segment_len=3)  # lower penalty for K=10 anchor space
    if not cps:
        continue
    
    # Use the most severe change point
    cp_ts, cp_severity = max(cps, key=lambda x: x[1])
    
    # Split trajectory at change point
    pre = [(ts, vec) for ts, vec in traj if ts < cp_ts]
    post = [(ts, vec) for ts, vec in traj if ts >= cp_ts]
    
    if len(pre) < 3 or len(post) < 2:
        continue
    
    try:
        cf = cvx.counterfactual_trajectory(pre, post, cp_ts)
        counterfactual_results.append({
            'user_id': uid,
            'change_point': cp_ts,
            'severity': cp_severity,
            'total_divergence': float(cf['total_divergence']),
            'max_divergence': float(cf['max_divergence_value']),
            'max_div_time': int(cf['max_divergence_time']),
            'n_pre': len(pre),
            'n_post': len(post),
        })
    except Exception as e:
        pass

cf_df = pd.DataFrame(counterfactual_results)
if cf_df.empty:
    print('No counterfactual results (no change points detected)')
else:
    print(f'{len(cf_df)} users with counterfactual analysis')
    print(f'Mean total divergence: {cf_df["total_divergence"].mean():.4f}')
    print(f'Mean max divergence:   {cf_df["max_divergence"].mean():.4f}')
    print(f'Mean CP severity:      {cf_df["severity"].mean():.4f}')


No counterfactual results (no change points detected)

In [16]:
# Visualize counterfactual for the user with highest divergence
if not cf_df.empty:
    top_user = cf_df.loc[cf_df['total_divergence'].idxmax()]
    uid = top_user['user_id']
    cp_ts = int(top_user['change_point'])
    traj = user_anchor_trajs[uid]
    
    pre = [(ts, vec) for ts, vec in traj if ts < cp_ts]
    post = [(ts, vec) for ts, vec in traj if ts >= cp_ts]
    cf = cvx.counterfactual_trajectory(pre, post, cp_ts)
    
    # Plot divergence curve
    div_ts = [t for t, _ in cf['divergence_curve']]
    div_vals = [v for _, v in cf['divergence_curve']]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=div_ts, y=div_vals,
        mode='lines+markers', name='Divergence',
        line=dict(color=C_DEP, width=2),
    ))
    fig.add_vline(x=cp_ts, line_dash='dash', line_color='gray',
                  annotation_text='Change Point')
    fig.update_layout(
        title=f'Counterfactual Divergence — User {uid} (severity={top_user["severity"]:.3f})',
        xaxis_title='Timestamp', yaxis_title='Distance (actual vs counterfactual)',
        height=400,
    )
    fig.show()

---
## 6. Summary Table

Compile all results for the paper.

In [17]:
summary = pd.DataFrame([
    {'Metric': 'Cohort mean drift (L2)', 'Depression': dep_report['mean_drift_l2'], 'Control': ctl_report['mean_drift_l2']},
    {'Metric': 'Cohort centroid drift', 'Depression': dep_report['centroid_l2'], 'Control': ctl_report['centroid_l2']},
    {'Metric': 'Dispersion change', 'Depression': dep_report['dispersion_change'], 'Control': ctl_report['dispersion_change']},
    {'Metric': 'Convergence score', 'Depression': dep_report['convergence_score'], 'Control': ctl_report['convergence_score']},
    {'Metric': 'Mean convergence windows', 'Depression': dep_df['n_windows'].mean(), 'Control': ctl_df['n_windows'].mean()},
    {'Metric': 'Temporal Join F1', 'Depression': np.mean(f1s), 'Control': '-'},
    {'Metric': 'Temporal Join AUC', 'Depression': np.mean(aucs), 'Control': '-'},
])
print(summary.to_string(index=False))

                  Metric  Depression   Control
  Cohort mean drift (L2)    0.047840  0.059932
   Cohort centroid drift    0.010436  0.005078
       Dispersion change    0.001843  0.000564
       Convergence score    0.207313  0.050829
Mean convergence windows    0.000000       0.0
        Temporal Join F1    0.264748         -
       Temporal Join AUC    0.500000         -
